# World Models 2018

这里，我们将复现经典论文：[World Models](https://arxiv.org/abs/1803.10122). 整个模型结构较为清晰，只需理解 Vision (V), Memory (M), 和 Controller (C) 这三个模块，整个模型就差不多已经理解了。

简单说一下整体架构。Vision 部分由 VAE 实现，负责将观测图像压缩到隐空间；Memory 部分由 MDN-RNN 实现，是整个世界模型的动力学模型（dynamic model），负责模拟隐空间中状态量的变化；Controller 是个简单的 MLP，负责根据当前 Memory 状态与当前 Vision 数据进行动作决策。我们预期在实现基本框架后在 `CarRacing-v0` 上面跑一跑，不涉及 `VizDoom` 相关实验。

## 1. 基本模块代码实现

这部分，我们分别实现 V, M, C 三个组件的代码。

### 1.1 Vision (VAE)

先来导入一些基本的库：

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

下面是 Encoder，参照原文附录的数据：

In [3]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=0), # -> (31, 31, 32)
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=0), # -> (14, 14, 64)
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=0), # -> (6, 6, 128)
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=0), # -> (2, 2, 256)
            nn.ReLU(),
        )

        self.fc_mu = nn.Linear(4 * 4 * 256, latent_dim)
        self.fc_logvar = nn.Linear(4 * 4 * 256, latent_dim)

    def forward(self, X):
        # X: (batch_size, 64, 64, 3)
        X = self.conv_layer(X)
        X_flatten = X.flatten(start_dim=1)

        mu = self.fc_mu(X_flatten)
        logvar = self.fc_logvar(X_flatten)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

紧接着是 Decoder，如下：

In [4]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        
        self.fc = nn.Linear(latent_dim, 1 * 1 * 1024)
        
        # Upsample + Conv2d
        self.upsample_layers = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(256, 128, kernel_size=4, padding=0),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128, 64, kernel_size=4, padding=0),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(64, 32, kernel_size=4, padding=0),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(32, 3, kernel_size=4, padding=0),
            nn.Sigmoid()
        )
    
    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 1, 1, 1024)
        x = self.upsample_layers(x)
        return x

### 1.2 Memory